# 25 L_SM: the Standard Model Lagrangian engine
*The Standard Model built from its gauge group and field content, checked against measured numbers, and used as the control for L_NN (VAPMIP) and L_(I|O). Everything above the comparison section is ESTABLISHED physics. The comparison section measures the other Lagrangians' code; failed or unexpected results stay.*

In [1]:
import sys, os, math, numpy as np
sys.path.insert(0, os.path.expanduser('~/Projects/ThePlace'))
from ValaQuenta.modules.lagrangian_sm import maths as S, compare as C
print(S.verify())

{'anomalies_cancel': True, 'su2_closure': 0.0, 'su3_closure': 1.1102230246251565e-16, 'su3_jacobi': np.float64(3.3306690738754696e-16), 'boson_masses_match': True, 'parameters': (19, 26, 28), 'ckm_unitarity': 1.1102230246251565e-16, 'm_h2_identity': True}


## What is derived and what is measured
The compact form `L = -1/4 F² + ψ̄ iD ψ + |DH|² - V + Yukawa` follows from the gauge group, the multiplets and renormalisability. The group and the multiplets are observed. The hypercharges are *derived* (below). The 19 numbers are *measured*.

In [2]:
a = S.anomaly_check(); print('anomaly sums:', {k: str(v) for k, v in a.items() if k in ('U1^3','SU3^2 U1','SU2^2 U1','grav^2 U1')}, ' Witten ok:', a['witten_ok'])
h = S.hypercharges_from_anomalies(); print('hypercharges from anomaly cancellation, unique:', h['unique'], {k: str(v) for k, v in h['solutions'][0].items()})
print('equal to the observed values:', h['solutions'][0] == h['sm'])

anomaly sums: {'U1^3': '0', 'SU3^2 U1': '0', 'SU2^2 U1': '0', 'grav^2 U1': '0'}  Witten ok: True
hypercharges from anomaly cancellation, unique: True {'Y_Q': '1/6', 'Y_L': '-1/2', 'Y_u': '2/3', 'Y_d': '-1/3', 'Y_e': '-1'}
equal to the observed values: True


### Where the anomaly cancellation comes from
The fifteen Weyl fermions of a generation are exactly 5̄ ⊕ 10 of SU(5); the hypercharge is a traceless SU(5) generator on them. And the fractional quark charges follow from N_c = 3 once the lepton charge is −1.

In [3]:
e = S.su5_embedding(); print('matches the SM content:', e['matches_content'], '| traces', e['trace_5bar'], e['trace_10'], '| A(10)+A(5bar) =', e['SU5_cubed_anomaly'])
for n in (1, 2, 3, 4, 5): c = S.hypercharges_from_anomalies(n)['charges']; print('N_c=%d  Q_u=%s  Q_d=%s  proton(uud)=%s' % (n, c['Q_u'], c['Q_d'], c['proton_uud']))

matches the SM content: True | traces 0 0 | A(10)+A(5bar) = 0
N_c=1  Q_u=1  Q_d=0  proton(uud)=2
N_c=2  Q_u=3/4  Q_d=-1/4  proton(uud)=5/4
N_c=3  Q_u=2/3  Q_d=-1/3  proton(uud)=1
N_c=4  Q_u=5/8  Q_d=-3/8  proton(uud)=7/8
N_c=5  Q_u=3/5  Q_d=-2/5  proton(uud)=4/5


## Group theory
Structure constants from the generators, Jacobi identity, Casimirs.

In [4]:
for k, v in S.group_checks().items(): print(k, {kk: (float(vv) if not isinstance(vv, int) else vv) for kk, vv in v.items()})

SU2 {'generators': 3, 'closure_residual': 0.0, 'jacobi_residual': 0.0, 'casimir_residual': 0.0, 'nonzero_f': 6}
SU3 {'generators': 8, 'closure_residual': 1.1102230246251565e-16, 'jacobi_residual': 3.3306690738754696e-16, 'casimir_residual': 2.220446049250313e-16, 'nonzero_f': 54}


## The number of parameters, counted
Complex 3×3 Yukawas minus the broken flavour symmetry, plus gauge, Higgs and θ_QCD.

In [5]:
for n in ('massless', 'dirac', 'majorana'): p = S.parameter_count(n); print('%-9s Yukawa real %2d  flavour generators %2d unbroken %d -> flavour physical %2d  TOTAL %d' % (n, p['yukawa_real'], p['flavour_generators'], p['unbroken_generators'], p['flavour_physical'], p['total']))

massless  Yukawa real 54  flavour generators 45 unbroken 4 -> flavour physical 13  TOTAL 19
dirac     Yukawa real 72  flavour generators 54 unbroken 2 -> flavour physical 20  TOTAL 26
majorana  Yukawa real 66  flavour generators 45 unbroken 1 -> flavour physical 22  TOTAL 28


## The Higgs sector, derived symbolically
|D H|² with H = (0, v/√2) gives the quadratic form of the four gauge fields; its Hessian eigenvalues are the squared masses.

In [6]:
import sympy as sp
s = S.ew_mass_matrix_symbolic(); print('quadratic form:', sp.simplify(s['quadratic_form']))
print('mass-matrix eigenvalues:', {str(k): v for k, v in s['eigenvalues'].items()})
e = S.higgs_potential_expansion(); print('V(h) coefficients:', {k: str(v) for k, v in e['coefficients'].items()}, ' m_h² =', e['m_h2'])

quadratic form: v**2*(g**2*(W1 - I*W2)*(W1 + I*W2) + (B*gp - W3*g)**2)/8
mass-matrix eigenvalues: {'g**2*v**2/4': 2, 'v**2*(g**2 + gp**2)/4': 1, '0': 1}
V(h) coefficients: {0: '-lam*v**4/4', 1: '0', 2: 'lam*v**2', 3: 'lam*v', 4: 'lam/4'}  m_h² = 2*lam*v**2


## Numbers against the PDG tables
Tree-level relations differ from measurement by the size of the loop corrections (a few tenths of a per cent), as they should.

In [7]:
h = S.higgs_sector(); print({k: round(v, 6) for k, v in h.items()})
t = S.ew_tree(); print({k: round(v, 5) for k, v in t.items()})
print('Yukawas:', {k: round(v, 5) for k, v in S.yukawas().items()})

{'m_h': 125.13, 'v': 246.21964, 'lam': 0.129136, 'mu2': 7828.75845, 'depth_GeV4': 118652880.785095, 'trilinear': 31.795833, 'quartic': 0.032284}


{'e': 0.31339, 'g': 0.65173, 'gp': 0.35742, 'm_W_tree': 80.23486, 'm_Z_tree': 91.50863, 'rho_tree': 1.0, 'm_W_pdg': 80.362, 'm_Z_pdg': 91.1879, 'm_W_diff_pct': -0.15821, 'm_Z_diff_pct': 0.35172}


Yukawas: {'u': 1e-05, 'd': 3e-05, 's': 0.00053, 'c': 0.00731, 'b': 0.02404, 't': 0.99136, 'e': 0.0, 'mu': 0.00061, 'tau': 0.01021}


## Flavour mixing and Z couplings

In [8]:
V = S.ckm_matrix(); print('|CKM|:\n', np.abs(V).round(4)); print('unitarity residual', S.unitarity_residual(V), ' J =', S.jarlskog(V))
print('PMNS J =', S.jarlskog(S.pmns_matrix()))
for k, z in S.z_couplings().items(): print(k, {kk: round(vv, 4) for kk, vv in z.items()})

|CKM|:
 [[0.9743 0.225  0.0037]
 [0.2249 0.9735 0.0418]
 [0.0086 0.0411 0.9991]]
unitarity residual 1.1102230246251565e-16  J = 3.1169935287554706e-05
PMNS J = -0.009793886924193728
nu {'T3': 0.5, 'Q': 0.0, 'g_L': 0.5, 'g_R': -0.0, 'g_V': 0.5, 'g_A': 0.5}
e {'T3': -0.5, 'Q': -1.0, 'g_L': -0.2688, 'g_R': 0.2312, 'g_V': -0.0376, 'g_A': -0.5}
u {'T3': 0.5, 'Q': 0.6667, 'g_L': 0.3459, 'g_R': -0.1541, 'g_V': 0.1917, 'g_A': 0.5}
d {'T3': -0.5, 'Q': -0.3333, 'g_L': -0.4229, 'g_R': 0.0771, 'g_V': -0.3459, 'g_A': -0.5}


## Running, and the failure of the couplings to meet
One loop: the three inverse couplings cross pairwise at three different scales. A spread, not a point. (Two-loop and threshold corrections move the numbers, not the conclusion.)

In [9]:
c = S.gauge_crossings(); print({k: ('%.3e' % v if k != 'log10_spread' else round(v, 2)) for k, v in c.items()})
print(S.vertex_inventory())

{'mu_12': '1.031e+13', 'mu_23': '9.710e+16', 'mu_13': '2.430e+14', 'log10_spread': 3.97}
{'bosonic_electroweak': 12, 'bosonic_qcd': 2, 'photon_fermion': 9, 'Z_fermion': 12, 'W_lepton': 3, 'W_quark_CKM_entries': 9, 'Higgs_fermion': 9, 'gluon_quark_flavours': 6, 'total_vertex_types': 62, 'note': 'per generation and colour structure; the components multiply these: 8 gluons, 3 colours, 3 generations.'}


## Anomalies chased to their source
1. Tree-level m_W, m_Z differ from PDG by 0.16 % and 0.35 %: the electroweak radiative correction Δr, computed below from the on-shell relation. 2. The one-loop λ → 0 scale is extremely sensitive to y_t(M_Z): not a number to quote.

In [10]:
import math
alpha0 = 1/137.035999; GF = S.INPUTS['G_F'][0]; mW = S.pdg_mass('W+')[0]; mZ = S.pdg_mass('Z0')[0]
A0 = math.pi*alpha0/(math.sqrt(2)*GF); s2 = 1 - mW**2/mZ**2; print('on-shell sin2 = %.5f (MSbar %.5f);  Delta r = %.4f  (literature about 0.036)' % (s2, S.INPUTS['sin2_thetaW_MSbar_MZ'][0], 1 - A0/(mW**2*s2)))
from scipy.integrate import solve_ivp
M0 = S.pdg_mass('Z0')[0]; k = 1/(16*math.pi**2); g1, g2, g3 = S.gauge_initial(); lam0 = S.higgs_sector()['lam']
def cross(yt0):
    def f(t, y):
        a1, a2, a3, yy, l = y; gp2 = 3/5*a1**2
        return [k*41/10*a1**3, -k*19/6*a2**3, -k*7*a3**3, k*yy*(4.5*yy**2 - 8*a3**2 - 9/4*a2**2 - 17/20*a1**2), k*(24*l**2 - 6*yy**4 + 12*l*yy**2 - 3*l*(3*a2**2 + gp2) + 3/8*(2*a2**4 + (a2**2 + gp2)**2))]
    ev = lambda t, y: y[4]; ev.terminal = True; ev.direction = -1
    s = solve_ivp(f, (0, math.log(1e19/M0)), [g1, g2, g3, yt0, lam0], events=ev, rtol=1e-9, atol=1e-12)
    return M0*math.exp(s.t_events[0][0]) if len(s.t_events[0]) else None
for yt in (0.92, 0.94, 0.96, 1.00): c = cross(yt); print('y_t(M_Z) = %.2f  lambda crosses zero at %s' % (yt, ('%.2e GeV' % c) if c else 'no scale below 1e19 GeV'))

on-shell sin2 = 0.22335 (MSbar 0.23122);  Delta r = 0.0364  (literature about 0.036)


y_t(M_Z) = 0.92  lambda crosses zero at no scale below 1e19 GeV
y_t(M_Z) = 0.94  lambda crosses zero at 1.42e+10 GeV
y_t(M_Z) = 0.96  lambda crosses zero at 2.07e+07 GeV
y_t(M_Z) = 1.00  lambda crosses zero at 1.66e+05 GeV


## The hat that the path-of-action tools use
The SM Higgs hat as a model dict: brim radius v, depth λ v⁴/4.

In [11]:
m = S.hat_model(); print({k: v for k, v in m.items() if k not in ('V',)})

{'model': 'standard_model', 'mu2': 7828.758449999999, 'lam': 0.12913605313016874, 'brim_radius': 246.21964023926205, 'depth': 118652880.78509493, 'm_over_v': 0.5082047877188265, 'tilt': 0.0, 'scale': 1.0}


## Comparison with L_NN (VAPMIP) as coded
Each term is evaluated at a field configuration and again with one field doubled; the exponent n in `term ∝ amplitude^n` identifies the interaction. A term-for-term isomorphism would give the same exponents.

In [12]:
ex = C.measured_exponents()
for k, v in ex.items(): print('%-26s' % k, v)

gauge kinetic              {'SM': {'A': 4.0}, 'NN': {'A': 2.0}}
matter (fermion-gauge)     {'SM': {'psi': 2.0, 'A': 1.0}, 'NN': {'psi': 2.0, 'A': 2.0}}
bias / Higgs potential     {'SM': {'identical_to_NN_as_coded_for_mu2_positive': True, 'max_difference': 5.551115123125783e-17}, 'NN': {'same_numbers': True}}
coupling (Yukawa)          {'SM': {'psi': 2.0, 'h': 1.0}, 'NN': {'psi': 1.0, 'beta': 1.0}}


### Two recorded findings on the coded L_NN
The hat term is exactly the Standard Model potential (for μ² > 0). `polar_lagrangian` does not apply the 1/φ to L_bias that its docstring writes, and μ² < 0 (the docstring's symmetry-breaking sign) gives a bowl, not a hat.

In [13]:
print(C.bias_conventions()); r = C.one_over_phi_on_bias(); print('1/phi applied to L_bias in polar_lagrangian:', r['applied'], '| integrand %.6f vs with 1/phi %.6f' % (r['integrand'], r['integrand_with_1_over_phi_on_bias']))

{'mu2=+1 (as coded)': 1.4142135623730951, 'mu2=-1 (docstring convention, as coded)': None, 'reading': "hat for mu^2 > 0; the docstring's mu^2 < 0 gives a bowl, no brim"}
1/phi applied to L_bias in polar_lagrangian: False | integrand 0.239196 vs with 1/phi 0.140463


## L_(I|O) as defined (∫ J_red J_blue ds)
The integrand is constant in σ: it carries no information about where on the path it is evaluated. The original L_dynamic (2026-06-12) was described as having local more/less action along the path, which this integrand does not have.

In [14]:
li = C.lio_integrand(1.0); print('J_red*J_blue constant in sigma:', li['constant'], ' value', round(li['value'], 6), '= exp(-E);  min(J_red,J_blue) peaks at sigma =', li['min_peaks_at'])

J_red*J_blue constant in sigma: True  value 0.367879 = exp(-E);  min(J_red,J_blue) peaks at sigma = 0.5


## The four rows side by side

In [15]:
import json
for r in C.comparison_table(): print(r['name'], '|', r['status'], '| free numbers:', r['free_parameters'])

L_SM | ESTABLISHED (form from symmetry + content + renormalisability; numbers measured) | free numbers: 19
L_NN / L_vap (VAPMIP, SMNNIP) | THEORETICAL | free numbers: 4
L_(I|O)  (the Pathway; deprecated alias L_dynamic) | CONJECTURE (definition); algebra ESTABLISHED | free numbers: 1
L_dynamic (original, 2026-06-12; Ainulindale wiki 51 'The Dynamic Geodesic') | CONJECTURE; its established home is mechanics in a time-dependent potential, dE/dt = dV/dt at fixed q (measured: witches_hat.action.moving_hat_exchange). Wiki 52 §3 (same day) describes it in words and lists 'define L_dynamic formally' as open; the rename to L_(I|O) on 2026-06-17 dropped L_standard and the time derivative. | free numbers: None


## Claim ledger
| claim | tier | where |
|---|---|---|
| anomalies cancel; hypercharges unique | ESTABLISHED | anomaly_check, hypercharges_from_anomalies |
| 19 / 26 / 28 parameters | ESTABLISHED | parameter_count |
| m_W, m_Z, couplings from the symbolic derivation | ESTABLISHED | ew_mass_matrix_symbolic |
| gauge couplings do not meet at one loop | ESTABLISHED | gauge_crossings |
| L_bias of L_NN = SM Higgs potential (μ² > 0) | ESTABLISHED (identity, measured) | measured_exponents |
| the other three L_NN terms have different degree from the SM terms | ESTABLISHED (measured on the code) | measured_exponents |
| 1/φ not applied to L_bias; μ² < 0 is a bowl | ESTABLISHED (code reading) | one_over_phi_on_bias, bias_conventions |
| L_(I|O) integrand is constant | ESTABLISHED (algebra) | lio_integrand |